# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

##Rules
I developed a simple rule-based scoring system using page freshness, search visibility, and user engagement. Each page receives a score based on these signals, and pages with higher scores are ranked as stronger candidates for content review and refresh. The ranked results are exported to work/outputs/baseline_action_score.csv.

If days_since_last_update > 365
AND impressions_last_30d > 300
AND engagement_rate < 50%

→ Review and Refresh Content
##Reason Codes
OLD_PAGE: The content has not been refreshed for a long time and may contain outdated information.

LOW_ENGAGEMENT: The page receives impressions but has a below-average click-through rate, suggesting low user engagement.

HIGH_VISIBILITY: The page appears frequently in search results and has strong visibility, making improvements potentially impactful.

CONTENT_REVIEW: Multiple indicators suggest that the page should be reviewed and considered for content refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone the repository if running in Google Colab
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Show the unit of analysis
print("Dataset shape:", df.shape)
df.head()

import os

# Create baseline score

df["baseline_score"] = (
    (df["days_since_last_update"] > 365).astype(int) * 2 +
    (df["impressions_last_30d"] > 300).astype(int) * 2 +
    (df["engagement_rate"] < 0.50).astype(int) * 2
)

# Generate reason codes
def get_reason(row):
    reasons = []

    if row["days_since_last_update"] > 365:
        reasons.append("OLD_PAGE")

    if row["impressions_last_30d"] > 300:
        reasons.append("HIGH_VISIBILITY")

    if row["engagement_rate"] < 0.50:
        reasons.append("LOW_ENGAGEMENT")

    if len(reasons) >= 2:
        reasons.append("CONTENT_REVIEW")

    return ", ".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

# Action label
df["action"] = "Review and Refresh Content"

# Sort pages
baseline = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Display Top 20
baseline.head(20)

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
29990,content_b7fa2d76e5ae,client_4e07408562,70.0,0.00,LOW,0.00,keyword article,informational,2501.0,15617.0,...,0.00,0.00,0.00,moderate,page_3_5,stable,-17.8,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,0.00,0.00,0.00,good,page_1,down,-66.2,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
16,content_78bd1d4a1d4d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,8200.0,52393.0,...,0.21,13.83,0.00,good,page_1,down,-39.1,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
26,content_72c5c2d73e5a,client_4e07408562,0.0,0.00,LOW,0.00,keyword article,informational,2686.0,17181.0,...,0.00,11.11,0.00,moderate,page_3_5,stable,-9.2,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
31,content_24ee79621dbf,client_19581e27de,50.0,0.09,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,0.00,moderate,page_3_5,up,42.5,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
28096,content_54289eb1f9e1,client_e629fa6598,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.00,14.29,0.00,moderate,striking,up,23.2,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
28100,content_bf84f3c7b54b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5992.0,39187.0,...,0.00,1.10,1.85,good,striking,up,72.7,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
28104,content_e8625556b0f8,client_19581e27de,50.0,0.07,LOW,7.04,keyword article,transactional,3064.0,22537.0,...,0.00,0.00,0.00,good,page_1,down,-46.8,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
28107,content_3918aaea55e2,client_8b940be7fb,0.0,0.00,LOW,0.00,keyword article,informational,3051.0,18935.0,...,0.00,0.00,0.00,moderate,striking,up,40.9,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content
28093,content_c8bc939f55d8,client_19581e27de,140.0,0.01,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,0.00,moderate,striking,down,-29.7,4,"HIGH_VISIBILITY, LOW_ENGAGEMENT, CONTENT_REVIEW",Review and Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| #  | Action          | Reason Code                                                | Confidence Note                                                                                      | What would make it wrong                                                       |
| -- | --------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ |
| 1  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because the page is old, has low CTR and enough impressions to justify optimization. | If the content was recently updated but the dataset has not reflected it yet.  |
| 2  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence due to declining CTR despite consistent search visibility.                           | If impressions are seasonal and expected to recover naturally.                 |
| 3  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong candidate because traffic potential exists but users are not clicking.                        | If the query intent has changed and requires a completely new page.            |
| 4  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because refreshing titles and content could improve CTR.                             | If ranking dropped because of technical SEO issues instead of content quality. |
| 5  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good refresh opportunity with sufficient historical impressions.                                     | If impressions come from irrelevant keywords.                                  |
| 6  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Page is old and underperforming compared to its visibility.                                          | If competitors recently dominated the SERP.                                    |
| 7  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because existing authority can be improved with fresh content.                       | If low CTR is caused by poor ranking rather than snippet quality.              |
| 8  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong refresh candidate based on multiple negative engagement signals.                              | If search demand has permanently declined.                                     |
| 9  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Page shows optimization potential without requiring new content creation.                            | If the page already matches current user intent perfectly.                     |
| 10 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because freshness is likely affecting performance.                                   | If CTR is limited by branded competitors.                                      |
| 11 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Consistent with the baseline rule for content refresh.                                               | If impressions are inflated by temporary events.                               |
| 12 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good candidate for title and metadata optimization.                                                  | If metadata has already been recently tested.                                  |
| 13 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Existing visibility indicates refresh may yield gains.                                               | If Google is testing new rankings temporarily.                                 |
| 14 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because several rule conditions are satisfied.                                       | If low CTR is due to SERP features rather than content.                        |
| 15 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Refreshing could improve both CTR and engagement.                                                    | If users already find answers directly in search results.                      |
| 16 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong rule match with clear optimization opportunity.                                               | If the page targets outdated keywords.                                         |
| 17 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good candidate based on historical performance decline.                                              | If traffic loss is caused by indexing issues.                                  |
| 18 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence due to multiple supporting signals.                                                  | If recent algorithm updates temporarily affected rankings.                     |
| 19 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Baseline rule strongly recommends refresh.                                                           | If the content is intentionally evergreen and unchanged.                       |
| 20 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Overall score suggests worthwhile refresh effort.                                                    | If another page on the site now satisfies the same search intent.              |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##Weak Picks
Several lower-ranked pages have missing values or very small traffic volumes, making their scores less reliable. Some recommendations could also be influenced by seasonal search behavior instead of outdated content. These pages require manual validation before any optimization decision is made.

##Leakage Check
The baseline rule only relies on historical content and performance metrics that would have been known when making the decision. No future-window metrics, product outcome flags, or the is_declining_label column were included in the scoring process. Therefore, the baseline ranking is free from target leakage and represents an honest decision-support system.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.